# 02 - Baseline B1: tonic-normalised pitch histogram + logistic regression

CPU runtime is enough. Needs `splits/v1.csv` from notebook 01 in persistent storage. Uses the tonic shipped with the
dataset, so results are an *upper bound* for the tonic step: the app will have to estimate the tonic itself.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/vishwesh-pujari/raaga-detector.git"
REPO_DIR = "/content/raaga-detector" if os.path.exists("/content") else "/kaggle/working/raaga-detector"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs are not always visible to a running kernel

try:  # Colab only: persistent storage for caches/splits/results
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
from raaga import env
from raaga.data import catalog, hmd, saraga
from raaga.features import cache as C
from raaga.models import baseline

RAW, ROOT = env.raw_data_home(), env.root()
split = pd.read_csv(ROOT / "splits" / "v1.csv")
INCLUDE_SARAGA = (split.dataset == "saraga").any()
print(len(split), "recordings,", split.raga.nunique(), "ragas; Saraga:", INCLUDE_SARAGA)

## 1. Features (built once, cached)
If the cache is complete this cell downloads nothing.

In [ ]:
TONIC = "tonic"  # HMD tonic file: "tonic" (automatic) or "tonicFine" (manually fine-tuned). Run both and compare.
PARAMS = dict(n_bins=120, chunk_s=30.0, hop_s=15.0, smooth_bins=1.0)
cache = C.cache_dir(ROOT, variant=f"pitch_{TONIC}", **PARAMS)
todo = [u for u in split.uid if not C._file(cache, u).exists()]
print(len(todo), "recordings still to featurise")
if todo:
    hmd.download(RAW)
    if INCLUDE_SARAGA:
        saraga.download(RAW)
    paths = catalog.build_catalog(RAW, include_saraga=INCLUDE_SARAGA, tonic_kind=TONIC)[["uid", "pitch_path", "tonic_hz"]]
    C.build(split[split.uid.isin(todo)].merge(paths, on="uid"), cache, **PARAMS)
data = C.load(split, cache)
print(data.X.shape, "chunks x bins")

## 2. Cross-validation on folds 0-3 (test fold 4 untouched)

In [ ]:
fold_of = dict(zip(split.uid, split.fold))
rows = []
for c in [0.01, 0.1, 1.0, 10.0]:
    res = baseline.cross_validate(data, fold_of, folds=(0, 1, 2, 3), C=c)
    rows.append({"C": c, **{k: np.mean([r[k] for r in res]) for k in ["chunk_top1", "rec_top1", "rec_top3", "rec_macro_f1", "rec_ece"]},
                 "rec_top1_std": np.std([r["rec_top1"] for r in res])})
cv = pd.DataFrame(rows).round(3)
display(cv)
chance = 1 / split.raga.nunique()
print("chance level (rec top-1):", round(chance, 3))
best_C = float(cv.loc[cv.rec_top1.idxmax(), "C"])
print("best C:", best_C)

In [ ]:
# where does it go wrong? confusion pooled over the CV folds
res = baseline.cross_validate(data, fold_of, folds=(0, 1, 2, 3), C=best_C)
classes = res[0]["classes"]
conf = sum(r["confusion"] for r in res)
off = [(conf[i, j], classes[i], classes[j]) for i in range(len(classes)) for j in range(len(classes)) if i != j and conf[i, j] > 0]
print("most confused (count, true, predicted):")
for n, t, p in sorted(off, reverse=True)[:12]:
    print(f"  {n}x  {t} -> {p}")
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(conf, cmap="Blues")
ax.set_xticks(range(len(classes)), classes, rotation=90, fontsize=7)
ax.set_yticks(range(len(classes)), classes, fontsize=7)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); plt.tight_layout(); plt.show()

## 3. Final test (run ONCE, when B1 is frozen)
Trains on folds 0-3 and evaluates on the held-out fold 4. Set `RUN_TEST = True` deliberately.

In [ ]:
RUN_TEST = False
if RUN_TEST:
    tr = data.subset(split[split.fold != 4].uid)
    te = data.subset(split[split.fold == 4].uid)
    model = baseline.make_model(best_C).fit(tr.X, tr.y)
    out = baseline.evaluate(model, te)
    summary = {k: v for k, v in out.items() if k not in ("confusion", "classes")}
    summary.update(C=best_C, chance=chance)
    print(json.dumps(summary, indent=2))
    (ROOT / "results").mkdir(exist_ok=True)
    (ROOT / "results" / "b1_test.json").write_text(json.dumps(summary, indent=2))